# Cómo discretizar sin invertir $A$ ni emplear `ss`

*(Traducción a Python del livescript `paso_continuo_a_discreto.mlx`, usando `numpy`, `scipy` y `control`)*

Para un sistema LTI en tiempo continuo $\dot x = Ax+Bu$, la discretización exacta con periodo $T$ da lugar a $x_{k+1} = Fx_k + Gu_k$, con

$$F = e^{AT}, \qquad G = \int_0^T e^{A(T-\tau)}B\,d\tau$$

La fórmula de $G$ es útil sobre todo cuando $A$ no es invertible (si lo fuera, también podríamos calcular $G = A^{-1}(F-I)B$, pero esta versión integral vale siempre).

In [1]:
import numpy as np
import control as ct
from scipy.linalg import expm
from scipy.integrate import quad_vec

np.set_printoptions(precision=6, suppress=True)

In [2]:
A = np.array([[0, 1], [0, -50]])   # por ejemplo
B = np.array([[0], [120]])
T = 0.0001   # periodo del sistema discreto

F = expm(A * T)
F

array([[1.      , 0.0001  ],
       [0.      , 0.995012]])

Para $G$ definimos el integrando (ver la función `integrando` al final del notebook) y usamos `scipy.integrate.quad_vec`, el equivalente Python de `integral(f,t0,t1,'ArrayValued',1)` de MATLAB para integrandos matriciales/vectoriales.

In [3]:
def integrando(t, A, B, T):
    return expm(A * (T - t)) @ B

In [4]:
G, err = quad_vec(lambda t: integrando(t, A, B, T), 0, T)
G

array([[0.000001],
       [0.01197 ]])

Lo hacemos también usando `control.ss` + `control.c2d` (equivalente a `ss` + `c2d` de MATLAB) para comprobar. Da igual qué pongamos en $C$ y $D$ para este propósito (no afectan al cálculo de $F$ y $G$), así que usamos unos valores cualesquiera.

In [5]:
sys = ct.ss(A, B, [[1, 0]], 0)   # C y D dan igual
sysd = ct.c2d(sys, T)
sysd

StateSpace(
array([[1.      , 0.0001  ],
       [0.      , 0.995012]]),
array([[0.000001],
       [0.01197 ]]),
array([[1., 0.]]),
array([[0.]]),
dt=0.0001,
name='sys[0]$sampled', states=2, outputs=1, inputs=1)

Comprobamos que coinciden ambos métodos,

In [6]:
print('F coincide con sysd.A?', np.allclose(F, sysd.A))
print('G coincide con sysd.B?', np.allclose(G, sysd.B))

F coincide con sysd.A? True
G coincide con sysd.B? True
